# Fake News Detection System
**CS 6140 – Machine Learning | Northeastern University**  
**Team:** Ritik Pravin Mota · Steve George · Sia Sopori

---
This notebook implements a multi-model fake news detection pipeline:
1. **TF-IDF + Logistic Regression** — lightweight baseline
2. **LSTM** — sequential deep learning model
3. **RoBERTa** — fine-tuned transformer

Models are evaluated on the ISOT dataset and cross-tested on FakeNewsNet for generalizability.

In [ ]:
!pip install evaluate

In [ ]:
import pandas as pd
import numpy as np
import re
import pickle
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from torch.utils.data import Dataset
import evaluate
from sklearn.model_selection import train_test_split

## 1. Data Loading & Preprocessing

We use the **ISOT Fake News Dataset** (44,898 articles, 2016–2017):  
- 23,481 fake articles (52%) — sourced from sites flagged by PolitiFact  
- 21,417 real articles (48%) — sourced exclusively from Reuters  

**Key preprocessing steps:**
- Remove Reuters bylines (e.g. `WASHINGTON (Reuters) -`) — prevents source leakage bias
- Remove featured image metadata artifacts
- Strip URLs and normalize whitespace
- Combine title + body into a single `content` field

**Split:** 80% train / 10% validation / 10% test (stratified)

In [ ]:
fake = pd.read_csv("/kaggle/input/datasets/siasop/faketrue/Fake.csv")
real = pd.read_csv("/kaggle/input/datasets/siasop/faketrue/True.csv")
fake["label"] = 0
real["label"] = 1
df = pd.concat([fake, real]).sample(frac=1, random_state=42).reset_index(drop=True)

def clean_text(text):
    text = re.sub(r'^.*?\(Reuters\)\s*-\s*', '', text)
    text = re.sub(r'featured image.*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df["content"] = (df["title"] + " " + df["text"]).apply(clean_text)
df = df[df["content"].str.strip() != ""].reset_index(drop=True)

X = df["content"]
y = df["label"]
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)
print("Data ready.")

## 2. RoBERTa Fine-tuning

`roberta-base` is a 125M parameter transformer pretrained on 160GB of text.  
We add a classification head and fine-tune all parameters for binary fake/real classification.

**Training config:** 3 epochs · batch size 8 (grad accum 2 → effective 16) · lr warmup 500 steps · fp16

In [ ]:
tokenizer_roberta = AutoTokenizer.from_pretrained("roberta-base")
train_encodings = tokenizer_roberta(list(X_train), truncation=True, padding=True, max_length=512)
val_encodings   = tokenizer_roberta(list(X_val),   truncation=True, padding=True, max_length=512)
print("Tokenization done.")

In [ ]:
class NewsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

train_dataset = NewsDataset(train_encodings, y_train)
val_dataset   = NewsDataset(val_encodings, y_val)
print("Datasets ready.")

In [ ]:
model_roberta = AutoModelForSequenceClassification.from_pretrained("roberta-base", num_labels=2)
print("Model loaded.")

In [ ]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

model_roberta = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2
)
print("Model loaded.")

In [ ]:
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir="/kaggle/working/roberta_results",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    warmup_steps=500,
    weight_decay=0.01,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    logging_steps=50,
    fp16=True,
)

trainer = Trainer(
    model=model_roberta,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

checkpoint_path = "/kaggle/input/datasets/siasop/faketrue/checkpoint-2245-20260412T213241Z-3-001/checkpoint-2245"
trainer.train(resume_from_checkpoint=checkpoint_path)

In [ ]:
from sklearn.metrics import classification_report

predictions = trainer.predict(val_dataset)
y_pred_roberta = np.argmax(predictions.predictions, axis=-1)
print(classification_report(y_val, y_pred_roberta, target_names=["Fake", "Real"]))

In [ ]:
import shutil

# Save model and tokenizer
trainer.save_model("/kaggle/working/roberta_final")
tokenizer_roberta.save_pretrained("/kaggle/working/roberta_final")

# Zip it so you can download it
shutil.make_archive("/kaggle/working/roberta_final", 'zip', "/kaggle/working/roberta_final")
print("Model saved and zipped.")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Evaluate on test set
test_dataset = NewsDataset(
    tokenizer_roberta(list(X_test), truncation=True, padding=True, max_length=512),
    y_test
)

predictions_test = trainer.predict(test_dataset)
y_pred_test = np.argmax(predictions_test.predictions, axis=-1)

print(classification_report(y_test, y_pred_test, target_names=["Fake", "Real"]))

cm = confusion_matrix(y_test, y_pred_test)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Fake", "Real"])
disp.plot(cmap="Blues")
plt.title("RoBERTa — Test Set")
plt.show()

## 3. TF-IDF + Logistic Regression & LSTM

### TF-IDF + Logistic Regression (Baseline)
Converts each article into a 50,000-dimensional sparse vector using unigrams and bigrams.  
Simple but interpretable — serves as our performance floor.

### LSTM
Captures sequential word dependencies that TF-IDF misses.  
Architecture: `Embedding(128) → SpatialDropout → LSTM(128) → Dense(64) → Sigmoid`  
Trained with early stopping (patience=2) to prevent overfitting.

In [ ]:
import datasets
print(datasets.__version__)

In [ ]:
# Fix: import load_dataset (required for FakeNewsNet cross-dataset evaluation)
from datasets import load_dataset

In [ ]:
dataset2 = load_dataset("GonzaloA/fake_news")
print(dataset2)

In [ ]:
df_fnn = pd.DataFrame(dataset2['test'])
print(df_fnn['label'].value_counts())
print(df_fnn.head(3))

In [ ]:
# Check label mapping
print(df_fnn[df_fnn['label']==0]['title'].head(2))
print("---")
print(df_fnn[df_fnn['label']==1]['title'].head(2))

In [ ]:
import os
print(os.listdir("/kaggle/input/datasets/siasop/tfidflr"))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1,2), stop_words="english")
X_train_tfidf = tfidf.fit_transform(X_train)

lr2 = LogisticRegression(max_iter=1000)
lr2.fit(X_train_tfidf, y_train)

X_fnn_tfidf = tfidf.transform(X_fnn)
y_pred_lr_fnn = lr2.predict(X_fnn_tfidf)

print("TF-IDF + LR on FakeNewsNet:")
print(classification_report(y_fnn, y_pred_lr_fnn, target_names=["Fake", "Real"]))

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, SpatialDropout1D
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_VOCAB = 50000
MAX_LEN = 512

lstm_tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token="<OOV>")
lstm_tokenizer.fit_on_texts(X_train)

X_train_seq = pad_sequences(lstm_tokenizer.texts_to_sequences(X_train), maxlen=MAX_LEN, truncating='post')
X_val_seq   = pad_sequences(lstm_tokenizer.texts_to_sequences(X_val),   maxlen=MAX_LEN, truncating='post')

lstm_model = Sequential([
    Embedding(MAX_VOCAB, 128),
    SpatialDropout1D(0.2),
    LSTM(128, dropout=0.2, recurrent_dropout=0.2),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

lstm_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)
history = lstm_model.fit(
    X_train_seq, y_train,
    epochs=10,
    batch_size=128,
    validation_data=(X_val_seq, y_val),
    callbacks=[early_stop]
)

In [ ]:
X_fnn_seq = pad_sequences(lstm_tokenizer.texts_to_sequences(X_fnn), maxlen=MAX_LEN, truncating='post')
y_pred_lstm_fnn = (lstm_model.predict(X_fnn_seq) > 0.5).astype(int)

print("LSTM on FakeNewsNet:")
print(classification_report(y_fnn, y_pred_lstm_fnn, target_names=["Fake", "Real"]))

In [ ]:
# Prepare the test data
df_fnn_test = pd.DataFrame(dataset2['test'])
df_fnn_test["content"] = (df_fnn_test["title"] + " " + df_fnn_test["text"]).apply(clean_text)
df_fnn_test = df_fnn_test[df_fnn_test["content"].str.strip() != ""].reset_index(drop=True)

X_fnn = df_fnn_test["content"]
y_fnn = df_fnn_test["label"]

# Tokenize
fnn_encodings = tokenizer_roberta(list(X_fnn), truncation=True, padding=True, max_length=512)
fnn_dataset = NewsDataset(fnn_encodings, y_fnn)

# Predict
predictions_fnn = trainer.predict(fnn_dataset)
y_pred_fnn = np.argmax(predictions_fnn.predictions, axis=-1)

print(classification_report(y_fnn, y_pred_fnn, target_names=["Fake", "Real"]))

## 4. Model Evaluation — ISOT Test Set

We evaluate all three models on the held-out ISOT test set (4,489 articles).  
Confusion matrices show the breakdown of true/false positives and negatives.

In [ ]:
# TF-IDF + LR evaluation on ISOT test set (Steve George)
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

X_test_tfidf = tfidf.transform(X_test)
y_pred_lr_test = lr2.predict(X_test_tfidf)

print('=== TF-IDF + Logistic Regression — ISOT Test Set ===')
print(classification_report(y_test, y_pred_lr_test, target_names=['Fake', 'Real']))

cm_lr = confusion_matrix(y_test, y_pred_lr_test)
ConfusionMatrixDisplay(confusion_matrix=cm_lr, display_labels=['Fake', 'Real']).plot(cmap='Greens')
plt.title('TF-IDF + LR — ISOT Test Set')
plt.tight_layout()
plt.show()

In [ ]:
# LSTM evaluation on ISOT test set (Steve George)
from tensorflow.keras.preprocessing.sequence import pad_sequences

X_test_seq = pad_sequences(
    lstm_tokenizer.texts_to_sequences(X_test),
    maxlen=MAX_LEN, truncating='post'
)
y_pred_lstm_test = (lstm_model.predict(X_test_seq) > 0.5).astype(int)

print('=== LSTM — ISOT Test Set ===')
print(classification_report(y_test, y_pred_lstm_test, target_names=['Fake', 'Real']))

cm_lstm = confusion_matrix(y_test, y_pred_lstm_test)
ConfusionMatrixDisplay(confusion_matrix=cm_lstm, display_labels=['Fake', 'Real']).plot(cmap='Oranges')
plt.title('LSTM — ISOT Test Set')
plt.tight_layout()
plt.show()

## 5. LSTM Training Curves

Plotting accuracy and loss per epoch shows how the LSTM learned over time  
and whether early stopping triggered at the right point.

In [ ]:
# LSTM training history plots (Steve George)
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['accuracy'], label='Train Accuracy', marker='o')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy', marker='o')
axes[0].set_title('LSTM — Accuracy per Epoch')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['loss'], label='Train Loss', marker='o')
axes[1].plot(history.history['val_loss'], label='Val Loss', marker='o')
axes[1].set_title('LSTM — Loss per Epoch')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('LSTM Training Curves', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Model Comparison Summary

Despite RoBERTa's apparent superiority on ISOT (100% vs 98%),  
**all three models achieve identical 97% F1 on FakeNewsNet**, suggesting the gap  
reflects dataset-specific patterns rather than genuine language understanding differences.

In [ ]:
# Model comparison summary table (Steve George)
import pandas as pd

results = {
    'Model': ['TF-IDF + Logistic Regression', 'LSTM', 'RoBERTa'],
    'ISOT Accuracy': ['98%', '98%', '100%'],
    'ISOT F1': ['98%', '98%', '100%'],
    'FakeNewsNet Accuracy': ['97%', '97%', '97%'],
    'FakeNewsNet F1': ['97%', '97%', '97%']
}

df_results = pd.DataFrame(results)
print('=== Model Comparison Summary ===')
df_results

## 7. Explainability — SHAP & LIME

### SHAP (Global Feature Importance)
SHAP explains which words drive the TF-IDF model's decisions across all predictions.

**Fake news indicators:** emotional/informal language (`just`, `like`), clickbait terms (`video`, `watch`, `breaking`), partisan emotional triggers  
**Real news indicators:** formal attribution (`said`, `told reporters`), precise dates (`wednesday`, `thursday`), formal terminology (`minister`, `presidential`)  
**Source bias check:** `reuters` had SHAP coefficient 22.3 before cleaning — after cleaning this dropped, confirming preprocessing was effective.

### LIME (Instance-level Explanation)
LIME explains individual predictions — why a specific article was classified as fake or real.

In [ ]:
!pip install shap
import shap

explainer = shap.LinearExplainer(lr2, X_train_tfidf, feature_perturbation="interventional")
shap_values = explainer(tfidf.transform(X_test[:100]))

shap.summary_plot(shap_values, feature_names=tfidf.get_feature_names_out(), max_display=20)

In [ ]:
!pip install lime
from lime.lime_text import LimeTextExplainer

explainer_lime = LimeTextExplainer(class_names=["Fake", "Real"])

def predict_proba(texts):
    return lr2.predict_proba(tfidf.transform(texts))

sample = X_test.iloc[5]
exp = explainer_lime.explain_instance(sample, predict_proba, num_features=10)
exp.show_in_notebook()

In [ ]:
# Find a real news article (label=1)
real_idx = y_test[y_test == 1].index[0]
sample_real = X_test.loc[real_idx]

exp_real = explainer_lime.explain_instance(sample_real, predict_proba, num_features=10)
exp_real.show_in_notebook()

In [ ]:
!pip install gradio

In [ ]:
import os
print(os.listdir("/kaggle/working"))

In [ ]:
print(type(tfidf))
print(type(lr2))
print(type(lstm_tokenizer))
print(type(lstm_model))
print(type(model_roberta))
print(type(tokenizer_roberta))

In [ ]:
import gradio as gr
from tensorflow.keras.preprocessing.sequence import pad_sequences


In [ ]:
def predict_news(text):
    cleaned = clean_text(text)
    
    tfidf_input = tfidf.transform([cleaned])
    lr_prob = lr2.predict_proba(tfidf_input)[0]
    lr_label = "Fake" if lr_prob[0] > 0.5 else "Real"
    lr_confidence = float(max(lr_prob))
    
    lstm_seq = pad_sequences(lstm_tokenizer.texts_to_sequences([cleaned]), maxlen=MAX_LEN, truncating='post')
    lstm_prob = lstm_model.predict(lstm_seq, verbose=0)[0][0]
    lstm_label = "Fake" if lstm_prob < 0.5 else "Real"
    lstm_confidence = float(1 - lstm_prob) if lstm_prob < 0.5 else float(lstm_prob)
    
    device = next(model_roberta.parameters()).device
    inputs = tokenizer_roberta(cleaned, return_tensors="pt", truncation=True, max_length=512, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model_roberta(**inputs)
    roberta_prob = torch.softmax(outputs.logits, dim=1)[0]
    roberta_label = "Fake" if roberta_prob[0] > roberta_prob[1] else "Real"
    roberta_confidence = float(max(roberta_prob))
    
    return {
        f"TF-IDF + LR: {lr_label}": lr_confidence,
        f"LSTM: {lstm_label}": lstm_confidence,
        f"RoBERTa: {roberta_label}": roberta_confidence,
    }

demo = gr.Interface(
    fn=predict_news,
    inputs=gr.Textbox(lines=10, placeholder="Paste a news article here..."),
    outputs=gr.Label(num_top_classes=3),
    title="Fake News Detector",
    description="Paste a news article and see predictions from three models — TF-IDF + LR, LSTM, and RoBERTa.",
    examples=[
        ["BREAKING: Hillary Clinton Arrested for Treason - Watch the Video Now!!!"],
        ["The Federal Reserve raised interest rates by 25 basis points on Wednesday, citing continued concerns about inflation, according to officials who spoke at a press conference."]
    ]
)

demo.launch(share=True)

In [ ]:
import os
print(os.listdir("/kaggle/input/datasets/siasop/roberta-final"))

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer_roberta = AutoTokenizer.from_pretrained("/kaggle/input/datasets/siasop/roberta-final")
model_roberta = AutoModelForSequenceClassification.from_pretrained("/kaggle/input/datasets/siasop/roberta-final")
model_roberta.eval()
print("RoBERTa loaded!")